In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def main():

#define data set name
    file_2024 = ('Alberta_power_usage_2024.csv')
    file_2021 = ('Alberta_power_usage_2021.csv')


    data_load = False
    end = False
    user_data_history = []
    regs = ['Calgary', 'Central', 'Edmonton', 'Northeast', 'Northwest', 'South']
    saved_results = {}
    while end != True:
        print('\n'*2)
        print(
            '====================================================\n'
            'ALBERTA POWER ANALYSIS TOOL\n'
            '====================================================\n'
            '1. Load Power Data sets\n'
            '2. View dataset summary\n'
            '3. Convert to daily / back to hourly\n'
            '4. Filter and calculate (by region, date range)\n'
            '5. Generate visualizations\n'
            '6. Compare 2021 vs 2024\n'
            '7. Export results\n'
            '8. Exit'
        ) 
        
           
        
        user_select = input("Enter your selection: ")
        while user_select not in ['1', '2', '3', '4', '5', '6', '7', '8']:
            print('Invalid selection')
            user_select = input('Enter your selection: ')

        print("\n"*2)
        

#option 1
        if user_select == '1' and data_load == True:
            print('Data already loaded')
        if user_select == '1' and data_load == False:
#user input year wanted
            data_year = int(input("what year of data(2021 or 2024)"))
            while data_year != 2021 and data_year != 2024:
                print("invalid input year")
                data_year = int(input("what year of data(2021 or 2024)"))

#tracks the history of the data loaded
            user_data_history.append(data_year)
#clean empty and duplacate spaces
            from PP3_Functions import data_clean
            if data_year == 2024:
                data_2024 = data_clean(file_2024)
            else:
                data_2021 = data_clean(file_2021)


#assignes the data set of the year selected
            if data_year == 2021:
                dataset = data_2021
            elif data_year == 2024:
                dataset = data_2024
    
#adjust for 2024 leap_year
            if data_year == 2024:
                leap_choice = input('Do you want to include febuary 29th?(yes/no)').lower()
                while leap_choice != 'yes' and leap_choice != 'no':
                    print('invalid choice please enter yes or no')
                    leap_choice = input('Do you want to include febuary 29th?(yes/no)').lower()
                if leap_choice == 'no':    
                    from PP3_Functions import leap_del
                    dataset_leapdel = leap_del(dataset)
                    dataset = dataset_leapdel

#ensure the data was loaded
            data_load = True
            hours = True
#option 2
        if user_select == '2' and data_load == True:
            if data_year == 2021: 
                leap = 'no leap'
            else:
                if leap_choice == 'no':
                    leap = 'not included'
                else:
                    leap = 'included'
            if hours == True:
                time_step = 'hours'
            else:
                time_step = 'days'     
            print(f"Year: {data_year} | leap day: {leap} | Time step: {time_step}")
            
            print()

            from PP3_Functions import region_stats
            print(f"{'Regions':<20}{'Mean':<10}{'Min':<10}{'Max':<10}")
            print('-'*50)
            rows = []
            for region in regs:
                average, minimum, maximum = region_stats(dataset, region)
                print(f"{region:<20}{average:<10.1f}{minimum:<10.1f}{maximum:<10.1f}")
                rows.append([region, average, minimum, maximum])
            
            summary = pd.DataFrame(rows, columns=['Region','Mean','Min','Max'])
            saved_results['summary']= summary





            reg_sums = []
            for clb in regs:     
                gp_sums = np.sum(dataset[clb])
                reg_sums.append(gp_sums)

            
            graph_sum_regs = np.array(regs.copy())
            graph_sum_sums = np.array(reg_sums)

            fig, ax = plt.subplots(figsize=(10,2.2))
            ax.bar(graph_sum_regs, graph_sum_sums, color="tab:orange")
            ax.set_xlabel("Region")
            ax.set_ylabel("Power (MW)")
            ax.set_title("Yearly power consumtion per region")
            ax.grid(True)
            plt.show()

                         
            
#option 3      
        if user_select == '3' and data_load == True:
#convert usage to by the day
            if hours == True:
                from PP3_Functions import day_choice
                conv_choice = day_choice('day')
                    
                if conv_choice.lower() == 'yes':
                    from PP3_Functions import to_day
                    dataset = to_day(dataset)
                    hours = False

            else:
                from PP3_Functions import day_choice
                conv_choice = day_choice('hour')
                
                if conv_choice.lower() == 'yes':
                    if data_year == 2024:
                        dataset = data_2024
                        if leap_choice == 'no':
                            dataset = dataset_leapdel
                    else:
                        dataset = data_2021
                    hours = True
                    

#option 4
        if user_select == '4' and data_load == True:
            from PP3_Functions import time_split
            restricted_data, start, end_date = time_split(dataset, hours, data_year)
            
            
            
            selected_regions = []
            while len(selected_regions) < 6:
                selected_region = input('Enter your desired regions(enter end to stop): ')
                if selected_region == 'end':
                    if len(selected_regions) == 0:
                        print('Select at least one region.')
                    else:
                        break
                elif selected_region not in regs:
                    print('Invalid region selection')

                elif selected_region in selected_regions:
                    print('Region already selected')
                    
                else: 
                    selected_regions.append(selected_region)
                
                
            filtered_data = restricted_data[['Days']+ selected_regions]


            valid_metric = ['average', 'minimum', 'maximum', 'total']
            metric = input('Enter a metric (average, minimum, maximum, total): ').lower()
            while metric not in valid_metric:
                print('Invalid metric')
                metric = input('Enter a metric (average, minimum, maximum, total): ').lower()
            results_rows = []
            for region in selected_regions:
                from PP3_Functions import calculate_metric
                value = calculate_metric(filtered_data, region, metric)
                results_rows.append([region, value])

            results_table = pd.DataFrame(results_rows, columns=['Region', metric])
            print()
            print(f'{metric} consumption from {start} to {end_date}')
            print(f"{'Region':<15}{metric:>15}")
            print('-'*34)
            for row in results_rows:
                print(f"{row[0]:<15}{row[1]:>15.1f}")

            saved_results['filtered'] = results_table
#option 5                
        if user_select == '5' and data_load == True:
            from PP3_Functions import time_split
            restricted_data_graph, start_graph, end_date_graph = time_split(dataset, hours, data_year)

            g_reg = []
            r_count = 1
            while len(g_reg) != 2:
                graph_region = input(f'Enter your desired regions({r_count}/2)')
                if graph_region not in regs:
                    print('Invalid region selection')
                    continue

                elif graph_region in g_reg:
                    print('Region already selected')
                    
                else: 
                    g_reg.append(graph_region)

                    r_count += 1


            graph_data_1 = np.array(restricted_data_graph[g_reg[0]].tolist())
            graph_data_2 = np.array(restricted_data_graph[g_reg[1]].tolist())
            time_graph = np.array(restricted_data_graph['Days'].tolist())

            fig, ax = plt.subplots(2,1, figsize=(10,5))
            ax[0].plot(time_graph, graph_data_1, color="tab:purple", linestyle='-', marker='o')
            ax[0].set_xlabel("Time (Days)")
            ax[0].set_ylabel("Power (MW)")
            ax[0].set_title(f"Power consumtion per day {g_reg[0]}")
            ax[0].grid(True)

            ax[1].plot(time_graph, graph_data_2, color="tab:green", linestyle='-', marker='x')
            ax[1].set_xlabel("Time (Days)")
            ax[1].set_ylabel("Power (MW)")
            ax[1].set_title(f"Power consumtion per day {g_reg[1]}")
            ax[1].grid(True)
            plt.tight_layout()
            plt.show()


#option 6        
        if user_select == '6' and data_load == True:
            if 2024 not in user_data_history:
                data_2024 = data_clean(data_2024)
                
            
            elif 2021 not in user_data_history: 
                data_2021 = data_clean(data_2021)




#option 7
        if user_select == '7' and data_load == True:
            export_choice_1 = input('Do you want to export the current dataset?(yes/no)').lower()
            while export_choice_1 != 'yes' and export_choice_1 != 'no':
                export_choice_1 = input('Do you want to export the current dataset?(yes/no)').lower()
            if export_choice_1 == 'yes':
                dataset.to_csv('Cleaned_Alberta_power_usage.csv',index=False)

            if 'summary' in saved_results:
                export_choice_2 = input('Do you want to export the dataset summary?(yes/no)').lower()
                while export_choice_2 != 'yes' and export_choice_2 != 'no':
                    export_choice_2 = input('Do you want to export the dataset summary?(yes/no)').lower()
                if export_choice_2 == 'yes':
                    saved_results['summary'].to_csv('Summary.csv',index=False)


            if 'filtered' in saved_results:
                export_choice_3 = input('Do you want to export the filtered data?(yes/no)').lower()
                while export_choice_3 != 'yes' and export_choice_3 != 'no':
                    export_choice_3 = input('Do you want to export the filtered data?(yes/no)').lower()
                if export_choice_3 == 'yes':
                    saved_results['filtered'].to_csv('Filtered.csv',index=False)
                
        
                
#option 8        
        
        if user_select == '8':
            end = True
            print('good bye!')
    

        if user_select not in '1' and user_select not in '8' and not data_load:
            print('Please load a dataset first')

if __name__ == "__main__":
    main()




ALBERTA POWER ANALYSIS TOOL
1. Load Power Data sets
2. View dataset summary
3. Convert to daily / back to hourly
4. Filter and calculate (by region, date range)
5. Generate visualizations
6. Compare 2021 vs 2024
7. Export results
8. Exit


Enter your selection:  1


what year of data(2021 or 2024) 2021


empty
DT_MST       0
Calgary      0
Central      0
Edmonton     0
Northeast    0
Northwest    0
South        0
dtype: int64



ALBERTA POWER ANALYSIS TOOL
1. Load Power Data sets
2. View dataset summary
3. Convert to daily / back to hourly
4. Filter and calculate (by region, date range)
5. Generate visualizations
6. Compare 2021 vs 2024
7. Export results
8. Exit


Enter your selection:  4


Enter the starting range date.(2021-01-01) 2021-01-01
Enter the ending range date.(2021-01-01) 2021-08-09
Enter your desired regions(enter end to stop):  Calgary
Enter your desired regions(enter end to stop):  Edmonton
Enter your desired regions(enter end to stop):  end
Enter a metric (average, minimum, maximum, total):  average



average consumption from 2021-01-01 to 2021-08-09
Region                 average
----------------------------------
Calgary                27828.3
Edmonton               36591.9



ALBERTA POWER ANALYSIS TOOL
1. Load Power Data sets
2. View dataset summary
3. Convert to daily / back to hourly
4. Filter and calculate (by region, date range)
5. Generate visualizations
6. Compare 2021 vs 2024
7. Export results
8. Exit


KeyboardInterrupt: Interrupted by user